# MD

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import os
import shutil
import MDAnalysis as mda

temperature = 1000 * kelvin
pdb_file = 'ala_no_water.pdb'
dcd_file = 'trajectory_tip3p.dcd'
log_file = 'md_log.txt'
timestep = 2 * femtosecond
from MDAnalysis.coordinates.DCD import DCDWriter

platform = Platform.getPlatformByName('CUDA')
properties = {'DeviceIndex': '0'}

pdb = PDBFile(pdb_file)
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addSolvent(forcefield, model='tip3p', padding=1.0*nanometer)
output_pdb = 'ala_with_water.pdb' 
with open(output_pdb, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)
    print(f"Saved solvated system to {output_pdb}")

system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME, nonbondedCutoff=1.0*nanometer, constraints=HBonds)
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
simulation = Simulation(modeller.topology, system, integrator, platform, properties)
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy()

simulation.reporters.append(StateDataReporter("md_log.txt", 1000, step=True,
        potentialEnergy=True, temperature=True, volume=True))
simulation.reporters.append(DCDReporter('trajectory_tip3p.dcd', 1000))


total_steps = 10_000_000  # 20 ns at 2 fs timestep

print("Running MD")
simulation.step(total_steps)


print("Simulation completed and all water-removed trajectories saved.")

In [ ]:
import MDAnalysis as mda
from MDAnalysis.coordinates.DCD import DCDWriter
from MDAnalysis.analysis import align

u = mda.Universe('ala_with_water.pdb', "trajectory_tip3p.dcd")

non_water = u.select_atoms("protein")

with DCDWriter("trajectory_no_water.dcd", n_atoms=non_water.n_atoms) as writer:
    for ts in u.trajectory:
        writer.write(non_water)

print("Water removed and new trajectory saved to 'trajectory_no_water.dcd'")

u_no_water = mda.Universe('ala_no_water.pdb', "trajectory_no_water.dcd")
reference = u_no_water.select_atoms("protein")
aligner = align.AlignTraj(u_no_water, reference, select="protein",
                         filename="trajectory_aligned_no_water.dcd")
aligner.run()